<div style="background-color:#EAF4EC; padding:20px; border-radius:10px;">

<h2 style="color:#2F6F4E; text-align:center; margin-bottom:5px;">
Forecast Integration & Model Selection
</h2>

<h4 style="color:#2F6F4E; text-align:center; margin-top:0;">
Master Thesis – ESG Governance Indicators (EU-27)
</h4>

<p style="font-size:14px; color:#2F6F4E;">
This notebook consolidates the forecasting outputs produced by the machine learning models
(<strong>XGBoost</strong> and <strong>Random Forest</strong>) and identifies the best-performing model for each ESG governance indicator.
</p>

<p style="font-size:14px; color:#2F6F4E;">
Using the out-of-sample evaluation metrics generated in the respective model notebooks,
the framework compares predictive performance based on <strong>RMSE</strong> and selects the model
with the lowest forecasting error for each indicator.
</p>

<p style="font-size:14px; color:#2F6F4E;">
The resulting forecasts are then combined into a unified dataset covering the full time horizon
<strong>(2000–2030)</strong>, integrating historical observations and machine learning projections.
This dataset constitutes the final input used for the subsequent stages of the research,
including clustering analysis, typology identification, and transition assessment across EU member states.
</p>

</div>

In [1]:
import os
import pandas as pd
files = os.listdir("../../data/processed/")
print(sorted(files))

['.DS_Store', 'RF_2000_2030.csv', 'XGB_2000_2030.csv', 'cluster_transitions.csv', 'data_final_EU27.csv', 'data_scaled_EU27.csv', 'df_forecast_all_indicators_EU27.csv', 'df_xy_pivoted.csv', 'esg_gov.csv', 'esg_gov17.csv', 'esg_gov17_eu27.csv', 'esg_gov17_eu27_since1996.csv', 'esg_gov17_eu27_since2000.csv', 'final_forecast_2000_2030.csv', 'final_forecast_2000_2030_scaled.csv', 'fuzzy_all.csv', 'metrics_rf.csv', 'metrics_xgb.csv', 'rf_full_2000_2030_clean.csv', 'xgb_forecasts_all_indicators.csv', 'xgb_full_2000_2030.csv', 'xgb_full_2000_2030_clean.csv', 'xgb_scaled_governance_EU27_2000_2030.csv']


In [2]:
# Ver o que está em cada ficheiro
df_fe_forecast   = pd.read_csv("../../data/processed/df_forecast_all_indicators_EU27.csv")
df_xgb_forecast  = pd.read_csv("../../data/processed/xgb_full_2000_2030_clean.csv")
df_rf_forecast   = pd.read_csv("../../data/processed/rf_full_2000_2030_clean.csv")

print("FE shape:", df_fe_forecast.shape)
print("XGB shape:", df_xgb_forecast.shape)
print("RF shape:", df_rf_forecast.shape)

print("\nFE columns:", df_fe_forecast.columns.tolist())
print("XGB columns:", df_xgb_forecast.columns.tolist())
print("RF columns:", df_rf_forecast.columns.tolist())

print("\nFE years:", df_fe_forecast["year"].unique() if "year" in df_fe_forecast.columns else "check col name")
print("XGB years:", df_xgb_forecast["year"].unique() if "year" in df_xgb_forecast.columns else "check col name")
print("RF years:", df_rf_forecast["year"].unique() if "year" in df_rf_forecast.columns else "check col name")

FE shape: (9384, 13)
XGB shape: (12555, 7)
RF shape: (12555, 7)

FE columns: ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', 'year', 'value', 'series_id', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_3', 'rolling_std_3', 'trend']
XGB columns: ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', 'year', 'pred_xgb', 'type']
RF columns: ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', 'year', 'pred_rf', 'type']

FE years: [2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013
 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023]
XGB years: [2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013
 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025 2026 2027
 2028 2029 2030]
RF years: [2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013
 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025 2026 2027
 2028 2029 2030]


In [3]:
df_xgb_forecast = pd.read_csv("../../data/processed/xgb_full_2000_2030_clean.csv")
df_rf_forecast  = pd.read_csv("../../data/processed/rf_full_2000_2030_clean.csv")

# métricas — precisas de guardar estes nos notebooks respectivos primeiro
results_xgb = pd.read_csv("../../data/processed/metrics_xgb.csv")
results_rf  = pd.read_csv("../../data/processed/metrics_rf.csv")

In [4]:
comparison = results_xgb[["Indicator Code", "RMSE_Test", "R2_Test"]].rename(
    columns={"RMSE_Test": "RMSE_XGB", "R2_Test": "R2_XGB"}
).merge(
    results_rf[["Indicator Code", "RMSE_Test", "R2_Test"]].rename(
        columns={"RMSE_Test": "RMSE_RF", "R2_Test": "R2_RF"}
    ),
    on="Indicator Code"
)

comparison["Best_Model"] = comparison[["RMSE_XGB", "RMSE_RF"]].idxmin(axis=1).str.replace("RMSE_", "", regex=False)

final_model_table = comparison[[
    "Indicator Code",
    "RMSE_XGB",
    "RMSE_RF",
    "R2_XGB",
    "R2_RF",
    "Best_Model"
]].sort_values("Indicator Code").reset_index(drop=True)

display(final_model_table)
# 2 — Construir dataset final
final_rows = []

for _, row in comparison.iterrows():
    code       = row["Indicator Code"]
    best_model = row["Best_Model"]

    if best_model == "XGB":
        df_source = df_xgb_forecast[df_xgb_forecast["Indicator Code"] == code].copy()
        df_source = df_source.rename(columns={"pred_xgb": "value"})
    else:
        df_source = df_rf_forecast[df_rf_forecast["Indicator Code"] == code].copy()
        df_source = df_source.rename(columns={"pred_rf": "value"})

    df_source["Best_Model"] = best_model
    final_rows.append(df_source)

df_final_forecast = pd.concat(final_rows, ignore_index=True)
df_final_forecast = df_final_forecast[["Country Name", "Country Code", "Indicator Name",
                                       "Indicator Code", "year", "value", "type", "Best_Model"]]

print("\nShape:", df_final_forecast.shape)
df_final_forecast.head(15)

,Indicator Code,RMSE_XGB,RMSE_RF,R2_XGB,R2_RF,Best_Model
0,CC.EST,0.082008,0.073368,0.987956,0.990360,RF
1,GB.XPD.RSDV.GD.ZS,0.073265,0.079112,0.993357,0.992254,XGB
2,GE.EST,0.131265,0.133377,0.945433,0.943663,XGB
3,IP.JRN.ARTC.SC,2781.811356,2767.578701,0.990461,0.990559,RF
4,IP.PAT.RESD,957.022056,1505.420235,0.985073,0.963064,XGB
5,IT.NET.USER.ZS,1.945054,1.925176,0.841135,0.844365,RF
6,NY.GDP.MKTP.KD.ZG,4.343923,2.807462,-1.546514,-0.063677,RF
7,PV.EST,0.101618,0.094289,0.778925,0.809663,RF
8,RL.EST,0.071830,0.066413,0.983719,0.986082,RF
9,RQ.EST,0.096613,0.086446,0.962702,0.970139,RF



Shape: (12555, 8)


,Country Name,Country Code,Indicator Name,Indicator Code,year,value,type,Best_Model
0,Austria,AUT,"School enrollment, primary and secondary (gros...",SE.ENR.PRSC.FM.ZS,2000,0.96969,observed,RF
1,Austria,AUT,"School enrollment, primary and secondary (gros...",SE.ENR.PRSC.FM.ZS,2001,0.96958,observed,RF
2,Austria,AUT,"School enrollment, primary and secondary (gros...",SE.ENR.PRSC.FM.ZS,2002,0.96443,observed,RF
3,Austria,AUT,"School enrollment, primary and secondary (gros...",SE.ENR.PRSC.FM.ZS,2003,0.96342,observed,RF
4,Austria,AUT,"School enrollment, primary and secondary (gros...",SE.ENR.PRSC.FM.ZS,2004,0.96304,observed,RF
5,Austria,AUT,"School enrollment, primary and secondary (gros...",SE.ENR.PRSC.FM.ZS,2005,0.96625,observed,RF
6,Austria,AUT,"School enrollment, primary and secondary (gros...",SE.ENR.PRSC.FM.ZS,2006,0.96986,observed,RF
7,Austria,AUT,"School enrollment, primary and secondary (gros...",SE.ENR.PRSC.FM.ZS,2007,0.96881,observed,RF
8,Austria,AUT,"School enrollment, primary and secondary (gros...",SE.ENR.PRSC.FM.ZS,2008,0.96560,observed,RF
9,Austria,AUT,"School enrollment, primary and secondary (gros...",SE.ENR.PRSC.FM.ZS,2009,0.96524,observed,RF


In [5]:
df_final_forecast.to_csv("../../data/processed/final_forecast_2000_2030.csv", index=False)
print("Saved final forecast to '../../data/processed/final_forecast_2000_2030.csv'")

Saved final forecast to '../../data/processed/final_forecast_2000_2030.csv'


In [6]:
# versão escalada — escalar por indicador
df_final_scaled = df_final_forecast.copy()
df_final_scaled["value_scaled"] = (
    df_final_scaled.groupby("Indicator Code")["value"]
    .transform(lambda x: (x - x.mean()) / x.std())
)
df_final_scaled.to_csv("../../data/processed/final_forecast_2000_2030_scaled.csv", index=False)
print("Saved scaled version to 'final_forecast_2000_2030_scaled.csv'")

Saved scaled version to 'final_forecast_2000_2030_scaled.csv'
